# COST — Controlled Gating Isolation

This notebook is a **clean add-on**. It does not rerun Initial COST, classical baselines, or earlier ablations.

**Experiment:** hold COST Improved fixed and isolate only controlled gating:
- `ON`: existing controlled salience gating.
- `OFF`: bypass gating (`x_gated = x_t`) and remove the gating sparsity penalty.

Safety:
- Training is OFF by default.
- First run only seed 42.
- Seeds 43–44 are enabled only after seed 42 is inspected.
- Outputs are saved under `revision2_outputs` (development folder name, kept for artifact traceability).

This notebook expects the same Google Drive project and existing processed NPZ used by notebook 01 ("Revision I" in code comments).

In [ ]:
# 1. SETUP + EXISTING CACHE
from pathlib import Path
import os, random, copy, time, json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import autocast, GradScaler

from sklearn.metrics import (
    roc_curve, roc_auc_score, average_precision_score, brier_score_loss
)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount:", e)

MIMIC_BASE = Path('/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1')
CACHE_FILE = MIMIC_BASE / 'mimic_cache' / 'hf_cohort_processed.npz'
OUT_DIR = MIMIC_BASE / 'revision2_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert CACHE_FILE.exists(), f"STOP: cache not found: {CACHE_FILE}"

d = np.load(CACHE_FILE, allow_pickle=True)
X_temporal = d['X_temporal']
X_static = d['X_static']
labels = d['labels']
years = d['years']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Cache loaded:", X_temporal.shape, X_static.shape, labels.shape)
print("Years:", years.min(), "-", years.max())
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUT_DIR)


## Important dependency note

To guarantee architectural identity with the main COST (Improved) model of notebook 01 ("Revision I" in code comments), this add-on should use the **exact existing definitions** of:
`ClinicalSalienceV2`, `SinusoidalPE`, `TemporalMultiHeadAttn`, `StaticTemporalFusion`, and the same split/loaders/configuration.

If you open this as a separate Colab runtime, paste/run the successful architecture + loader/config cells from notebook 01 before Cell 3 below. The assertion cell will stop safely if anything is missing; it will never silently invent configuration values.

In [ ]:
# 2. DEPENDENCY PREFLIGHT — NO TRAINING
required_globals = [
    'ClinicalSalienceV2', 'SinusoidalPE', 'TemporalMultiHeadAttn',
    'StaticTemporalFusion', 'loaders', 'CONFIG_V2',
    'N_INPUT', 'N_STATIC', 'SEED'
]
missing = [x for x in required_globals if x not in globals()]
print("Missing dependencies:", missing if missing else "NONE")
if missing:
    print("\nSTOP HERE. Run/copy the exact successful Revision-I/II definition cells for these items.")
else:
    print("✅ Exact project dependencies available. Safe to continue.")


In [ ]:
# 3. REVISION-II MODEL: ONLY CONTROLLED GATING CHANGES
# Run only if Cell 2 reports NONE.

class DSRNNv2_GatingIsolation(nn.Module):
    def __init__(self, n_input, n_static, hidden_size, n_lstm_layers,
                 n_heads, attention_dim, salience_hidden, dropout=0.3,
                 stoch_depth_p=0.1, lambda_sparse=5e-4,
                 use_controlled_gating=True):
        super().__init__()
        self.hidden_size = hidden_size
        self.lambda_sparse = lambda_sparse
        self.use_controlled_gating = bool(use_controlled_gating)

        self.input_proj = nn.Sequential(
            nn.Linear(n_input, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.pos_enc = SinusoidalPE(hidden_size)
        self.salience = ClinicalSalienceV2(
            hidden_size + hidden_size, salience_hidden, hidden_size
        )
        self.lstm = nn.LSTM(
            hidden_size, hidden_size, num_layers=n_lstm_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if n_lstm_layers > 1 else 0
        )
        self.bidir_proj = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU()
        )
        self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
        self.fusion = StaticTemporalFusion(
            hidden_size, n_static, attention_dim, n_heads, dropout
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(attention_dim),
            nn.Dropout(dropout),
            nn.Linear(attention_dim, attention_dim // 2),
            nn.GELU(),
            nn.Linear(attention_dim // 2, 1)
        )

    def forward(self, x_temporal, x_static, return_masks=False):
        batch, seq, _ = x_temporal.shape
        x_proj = self.pos_enc(self.input_proj(x_temporal))

        h_prev = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        gated_seq = []
        masks = []
        sparsity_sum = torch.tensor(0.0, device=x_temporal.device)

        for t in range(seq):
            x_t = x_proj[:, t, :]

            if self.use_controlled_gating:
                m_t, s_t = self.salience(x_t, h_prev)
                x_used = m_t * x_t + (1.0 - m_t) * x_t * 0.1
                sparsity_sum = sparsity_sum + m_t.norm(p=1)
                masks.append((m_t.detach() > 0.5).float().unsqueeze(1))
            else:
                x_used = x_t

            gated_seq.append(x_used.unsqueeze(1))
            h_prev = h_prev * 0.9 + x_used.detach() * 0.1

        x_seq = torch.cat(gated_seq, dim=1)

        if self.use_controlled_gating:
            sparsity_loss = self.lambda_sparse * sparsity_sum / (seq * batch)
        else:
            sparsity_loss = torch.tensor(0.0, device=x_temporal.device)

        lstm_out, _ = self.lstm(x_seq)
        lstm_out = self.bidir_proj(lstm_out)
        lstm_out, _ = self.temporal_attn(lstm_out)
        z, _ = self.fusion(x_static, lstm_out)
        logits = self.classifier(z).squeeze(-1)

        if return_masks:
            mask_tensor = torch.cat(masks, dim=1) if masks else None
            return logits, sparsity_loss, mask_tensor
        return logits, sparsity_loss

print("✅ DSRNNv2_GatingIsolation defined — NO TRAINING")


In [ ]:
# 4. MODEL BUILDER + STRUCTURAL SANITY — NO TRAINING

def build_gating_model(config, gating):
    return DSRNNv2_GatingIsolation(
        n_input=N_INPUT,
        n_static=N_STATIC,
        hidden_size=config['hidden_size'],
        n_lstm_layers=config['n_lstm_layers'],
        n_heads=config['n_heads'],
        attention_dim=config['attention_dim'],
        salience_hidden=config['salience_hidden'],
        dropout=config['dropout'],
        stoch_depth_p=config.get('stoch_depth_p', 0.1),
        lambda_sparse=config['lambda_sparse'],
        use_controlled_gating=gating
    )

model_on = build_gating_model(CONFIG_V2, True).to(DEVICE)
model_off = build_gating_model(CONFIG_V2, False).to(DEVICE)

n_on = sum(p.numel() for p in model_on.parameters())
n_off = sum(p.numel() for p in model_off.parameters())

print("ON params :", f"{n_on:,}")
print("OFF params:", f"{n_off:,}")
print("Parameter structure identical:", n_on == n_off)
assert n_on == n_off, "STOP: model structures differ unexpectedly."

# State-dict key check
assert list(model_on.state_dict().keys()) == list(model_off.state_dict().keys())
print("State-dict keys identical: True")
print("Only runtime switch differs:",
      model_on.use_controlled_gating, "vs", model_off.use_controlled_gating)
print("✅ Structural isolation check passed — NO TRAINING")


In [ ]:
# 5. FORWARD SANITY — NO TRAINING
batch = next(iter(loaders['train']))
x_temp, x_stat, y = batch
x_temp = x_temp[:4].to(DEVICE)
x_stat = x_stat[:4].to(DEVICE)

model_on.eval()
model_off.eval()
with torch.no_grad():
    out_on, sp_on, masks_on = model_on(x_temp, x_stat, return_masks=True)
    out_off, sp_off, masks_off = model_off(x_temp, x_stat, return_masks=True)

print("ON logits shape :", tuple(out_on.shape))
print("OFF logits shape:", tuple(out_off.shape))
print("ON sparsity penalty :", float(sp_on))
print("OFF sparsity penalty:", float(sp_off))
print("ON masks shape:", None if masks_on is None else tuple(masks_on.shape))
print("OFF masks:", masks_off)
assert out_on.shape == out_off.shape
assert float(sp_off) == 0.0
assert masks_off is None
print("✅ Forward sanity passed — NO TRAINING")


In [ ]:
# 6. METRICS + TRAINING UTILITIES

class FocalLossWithSmoothing_R2(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.alpha, self.gamma, self.smoothing = alpha, gamma, smoothing
    def forward(self, logits, targets):
        targets_smooth = targets * (1-self.smoothing) + 0.5*self.smoothing
        bce = F.binary_cross_entropy_with_logits(logits, targets_smooth, reduction='none')
        prob = torch.sigmoid(logits)
        p_t = prob*targets + (1-prob)*(1-targets)
        alpha_t = self.alpha*targets + (1-self.alpha)*(1-targets)
        return (alpha_t * (1-p_t)**self.gamma * bce).mean()

def rdrop_kl_r2(a, b):
    p1, p2 = torch.sigmoid(a), torch.sigmoid(b)
    eps=1e-8
    kl1=p1*(torch.log(p1+eps)-torch.log(p2+eps))
    kl2=p2*(torch.log(p2+eps)-torch.log(p1+eps))
    return ((kl1+kl2)*0.5).mean()

@torch.no_grad()
def eval_r2(model, loader):
    model.eval()
    ys, ps = [], []
    for xt, xs, y in loader:
        xt, xs = xt.to(DEVICE), xs.to(DEVICE)
        logits, _ = model(xt, xs)
        ys.extend(y.numpy())
        ps.extend(torch.sigmoid(logits).cpu().numpy())
    return np.asarray(ys), np.asarray(ps)

def metrics_r2(y, p):
    fpr,tpr,_=roc_curve(y,p)
    spec=1-fpr
    idx=np.argmin(np.abs(spec-0.90))
    return {
        'auc':roc_auc_score(y,p),
        'auprc':average_precision_score(y,p),
        'brier':brier_score_loss(y,p),
        'sens90':float(tpr[idx])
    }


In [ ]:
# 7. SAFETY SWITCH — TRAINING OFF BY DEFAULT
RUN_TRAINING = False
RUN_SEED42 = True
RUN_SEEDS_43_44 = False
SEEDS = [42,43,44]

print("RUN_TRAINING:", RUN_TRAINING)
print("Seed 42 first:", RUN_SEED42)
print("Seeds 43–44:", RUN_SEEDS_43_44)
print("🛑 No training until RUN_TRAINING=True manually.")


In [ ]:
# 8. TRAIN ONE MATCHED CONFIGURATION
# This function does nothing until explicitly called.

def train_gating_isolation(config, gating, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = build_gating_model(config, gating).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=config['lr'],
                      weight_decay=config['weight_decay'], betas=(0.9,0.999))
    total_steps = config['max_epochs'] * len(loaders['train'])
    scheduler = OneCycleLR(
        optimizer, max_lr=config['lr'], total_steps=total_steps,
        pct_start=config['warmup_pct'], anneal_strategy='cos'
    )
    focal = FocalLossWithSmoothing_R2(
        config['focal_alpha'], config['focal_gamma'], config['label_smoothing']
    )
    scaler = GradScaler()

    best_auc=-np.inf
    best_state=None
    patience=0
    history=[]

    for epoch in range(1, config['max_epochs']+1):
        model.train()
        losses=[]
        for xt,xs,y in loaders['train']:
            xt,xs,y=xt.to(DEVICE),xs.to(DEVICE),y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                l1,sp1=model(xt,xs)
                l2,sp2=model(xt,xs)
                task=(focal(l1,y)+focal(l2,y))*0.5
                rdrop=rdrop_kl_r2(l1,l2)
                sp=(sp1+sp2)*0.5
                loss=task + config['rdrop_alpha']*rdrop + sp
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))

        vy,vp=eval_r2(model,loaders['val'])
        va=roc_auc_score(vy,vp)
        history.append((epoch,np.mean(losses),va))
        print(f"epoch {epoch:03d} | loss={np.mean(losses):.5f} | val_auc={va:.5f}")

        if va>best_auc:
            best_auc=va
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience=0
        else:
            patience+=1
        if patience>=config['patience']:
            print("Early stopping:", epoch)
            break

    model.load_state_dict(best_state)
    model.to(DEVICE)
    ty,tp=eval_r2(model,loaders['test'])
    met=metrics_r2(ty,tp)

    tag=f"gating_{'ON' if gating else 'OFF'}_seed{seed}"
    torch.save(best_state, OUT_DIR/f"{tag}.pt")
    np.savez_compressed(OUT_DIR/f"{tag}_predictions.npz",
                        y_true=ty,y_prob=tp,seed=seed,controlled_gating=gating)
    pd.DataFrame(history,columns=['epoch','train_loss','val_auc']).to_csv(
        OUT_DIR/f"{tag}_history.csv",index=False)

    print("Saved:", tag)
    print(met)
    return model,met,ty,tp


## First actual run

Do **not** run the next cell until the structural and forward sanity checks above pass.

For the first timing/sanity experiment, train **gating OFF, seed 42 only**. This configuration was not part of the main COST (Improved) runs in notebook 01.

After it completes, inspect runtime, validation trajectory, test AUC/AUPRC/Brier/Sens@90%, and saved files before enabling seeds 43–44.

In [ ]:
# 9. FIRST NEW REVISION-II TRAINING — OFF / SEED 42 ONLY
if RUN_TRAINING and RUN_SEED42:
    t0=time.time()
    model_off_42, metrics_off_42, y42, p42 = train_gating_isolation(
        CONFIG_V2, gating=False, seed=42
    )
    print(f"Elapsed: {(time.time()-t0)/60:.1f} min")
else:
    print("🔒 Training skipped. Set RUN_TRAINING=True only after sanity checks are approved.")


In [ ]:
# 10. OPTIONAL SEEDS 43–44 — KEEP DISABLED UNTIL SEED 42 REVIEWED
results_off = []
if RUN_TRAINING and RUN_SEEDS_43_44:
    for seed in [43,44]:
        t0=time.time()
        model,met,y,p=train_gating_isolation(CONFIG_V2,gating=False,seed=seed)
        results_off.append({"seed":seed,**met})
        print(f"seed {seed} elapsed {(time.time()-t0)/60:.1f} min")
else:
    print("🔒 Seeds 43–44 disabled.")


## Next analyses after training
After the matched OFF runs are complete:
1. Compare against matched/existing ON runs with the same seeds.
2. Extract actual gating masks from final ON checkpoints.
3. Compute retention/sparsity distribution.
4. Benchmark parameters, memory, FLOPs, and latency.
5. Re-run per-seed statistical analysis.
6. Perform subject-ID overlap audit separately from the raw cohort metadata (no model retraining).

Do not change the manuscript's conclusions until these results are available.
